<a href="https://colab.research.google.com/github/EthanFBusiness/EthanFBusiness.github.io/blob/main/Minimising%20Personal%20Portfolio%20Volatility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study: Minimising Personal Portfolio Volatility
### An Investigation into Mean-Variance Optimisation and Estimation Error

## 1. Motivation
As a retail investor holding a concentrated portfolio of UK equities (SHEL, BP, HSBA, VOD),
I am exposed to significant sector-specific risk. While my natural instinct might be to
simply "split my money equally" (the $1/N$ strategy), my academic background in
Multivariate Calculus and Stochastic Modelling suggests a more rigorous approach.

The goal of this project is to apply Markowitz's Mean-Variance Theory to my own
holdings to find the "Global Minimum Variance" (GMV) weights. I want to answer one
fundamental question:

Can I use matrix algebra to significantly reduce my portfolio's volatility compared
to a naive equal-weight strategy?

## 2. The Theoretical Challenge
On paper, the solution is elegant. By minimising the quadratic form of the
variance-covariance matrix ($\Sigma$), we can find the exact point that offers the lowest possible risk.


However, as I will demonstrate, this optimisation method has its drawbacks.
I will investigate:
1. Does the maths pose a risk of providing negative portfolio weights?
2. The Estimation Error: Does a portfolio optimised for 2024 actually protect
me in 2025?

## 3. Theoretical Framework: Expected Return and Portfolio Risk

### 3.1 Portfolio Expected Return
A portfolio's expected return, denoted by $R_p$, is given by:

$$R_p = w_1\mu_1 + w_2\mu_2 + \dots + w_n\mu_n$$

For $n$ stocks, where $n \in \mathbb{N}$:
* $w_i$: proportion of portfolio held (weights) in stock $i$, for $i=1, \dots, n$
* $\mu_i$: expected return of stock $i$, for $i=1, \dots, n$

In matrix shorthand, let $\mathbf{w}$ be the $n \times 1$ weight matrix (vector):
$$\mathbf{w} = \begin{pmatrix} w_1 \\ \vdots \\ w_n \end{pmatrix}$$

Let $\boldsymbol{\mu}$ be the $n \times 1$ expected return matrix (vector):
$$\boldsymbol{\mu} = \begin{pmatrix} \mu_1 \\ \vdots \\ \mu_n \end{pmatrix}$$

We can then say the portfolio's expected return is:
$$R_p = \mathbf{w}^T \boldsymbol{\mu}$$


### 3.2 Portfolio Risk (Variance)
The risk of a portfolio is defined as its variance.

Recall that for a portfolio's "actual" return:
$$R_p = \sum_{i=1}^{n} w_i R_i$$

Where $R_i$ is the $i^{th}$ asset's actual return (a random variable).

By rewriting $\mathbf{w} = \begin{pmatrix} w_1 \\ \vdots \\ w_n \end{pmatrix}$ and $\mathbf{R} = \begin{pmatrix} R_1 \\ \vdots \\ R_n \end{pmatrix}$, we have:
$$R_p = \mathbf{w}^T \mathbf{R}$$

The variance is then derived as:
$$var(R_p) = var(\mathbf{w}^T \mathbf{R}) = \mathbf{w}^T var(\mathbf{R}) \mathbf{w}$$

Where $var(\mathbf{R})$ is the Covariance Matrix for the $i^{th}$ and $j^{th}$ stocks, usually denoted as $\Sigma$.

$$\Rightarrow \sigma^2_p = \mathbf{w}^T \Sigma \mathbf{w}$$

## 4. Risk Minimisation using Lagrange Multipliers
To minimise portfolio risk, we seek to minimise the term $w^T \Sigma w$. We must introduce a Lagrange Multiplier ($\lambda$) such that our weights sum to 1.

### 4.1 The Lagrangian Function
$$L(w, \lambda) = w^T \Sigma w - \lambda(w^T \mathbf{1} - 1)$$

Taking the partial derivative with respect to $w$:
$$\frac{\partial L}{\partial w} = 2\Sigma w - \lambda \mathbf{1}$$

Setting the derivative to zero to find the minimum:
$$2\Sigma w - \lambda \mathbf{1} = 0 \Rightarrow w = \frac{\lambda}{2} \Sigma^{-1} \mathbf{1}$$

This result shows that perfectly optimised weights are proportional to the inverse covariance matrix. This is intuitive, as a perfectly diversified portfolio would prioritise stocks with the most unique (lowest covariance/correlation) properties.

### 4.2 Solving for $w$
Since the sum of weights must equal 1 ($\mathbf{1}^T w = 1$), and noting that $\mathbf{1}^T w = \sum_{i=1}^n w_i = 1$:

$$\mathbf{1}^T \left( \frac{\lambda}{2} \Sigma^{-1} \mathbf{1} \right) = 1$$
$$\frac{\lambda}{2} (\mathbf{1}^T \Sigma^{-1} \mathbf{1}) = 1$$
$$\frac{\lambda}{2} = \frac{1}{\mathbf{1}^T \Sigma^{-1} \mathbf{1}}$$

Substituting back into the expression for $w$:
$$\Rightarrow w = \frac{1}{\mathbf{1}^T \Sigma^{-1} \mathbf{1}} \times \Sigma^{-1} \mathbf{1} = \frac{\Sigma^{-1} \mathbf{1}}{\mathbf{1}^T \Sigma^{-1} \mathbf{1}}$$

### Implementation Notes
* This analytical solution can lead to negative weights (short positions).
* Adding a constraint for all $w_i > 0$ (long-only) is mathematically more difficult and typically requires numerical optimisation methods.

## 5. Implementation: Testing the Analytical Solution in Python

Having derived the closed-form solution for the weight vector $\mathbf{w}^*$ using matrix calculus, I will now implement this logic using historical data for four UK equities: Shell (SHEL.L), BP (BP.L), HSBC (HSBA.L), and Vodafone (VOD.L).

At this stage, the implementation strictly follows the Lagrangian derivation, applying only the budget constraint (weights must sum to 1). This serves as a validation step to see if the theoretical "Global Minimum Variance" portfolio is actionable.

In [14]:
import yfinance as yf
import numpy as np
import pandas as pd

# 1. Download Historical Data (Training Period: 2023 - 2025)
tickers = ['SHEL.L', 'BP.L', 'HSBA.L', 'VOD.L']
df = yf.download(tickers, start="2023-01-01", end="2025-01-01", auto_adjust=True)

# Calculate logarithmic returns (Standard for matrix-based finance)
returns = np.log(df['Close'] / df['Close'].shift(1)).dropna()

# 2. Generate the Annualised Covariance Matrix (Sigma)
sigma = returns.cov() * 252

# 3. Apply the Analytical GMV Formula: w = (inv(Sigma) @ 1) / (1.T @ inv(Sigma) @ 1)
inv_sigma = np.linalg.inv(sigma)
ones = np.ones(len(tickers))

# Numerator and Denominator from the Lagrangian derivation
numerator = inv_sigma @ ones
denominator = ones.T @ inv_sigma @ ones
w_optimal = numerator / denominator

# Format results for analysis
theoretical_weights = pd.Series(w_optimal, index=tickers)
print("--- Theoretical Weights (Lagrangian Solution) ---")
print(theoretical_weights.round(4))

[*********************100%***********************]  4 of 4 completed

--- Theoretical Weights (Lagrangian Solution) ---
SHEL.L   -0.0148
BP.L      0.3430
HSBA.L    0.4224
VOD.L     0.2494
dtype: float64


## 6. Critical Analysis: The Short-Selling Problem

Upon inspecting the weights $\mathbf{w}^*$ generated by the analytical Lagrangian solution, a significant practical issue arises: the presence of negative weight values.

#### The Physical Reality of Stocks
In our mathematical derivation, a negative weight represents a short position. While this balances the portfolio variance mathematically, for a retail investor, this is often impossible or undesirable:

* Physical Ownership: As a retail investor, the primary goal is typically to physically own the underlying shares (Long-Only). A negative weight $w_i < 0$ implies selling shares we do not own.
* Brokerage Limits: Most standard retail accounts do not support short-selling UK equities, nor do they provide the margin required to maintain these positions.
* Risk Profile: Short-selling carries theoretically infinite risk if the stock price rises, which contradicts our objective of a Global Minimum Variance portfolio.

Because the closed-form Lagrangian solution:
$$\mathbf{w}^* = \frac{\Sigma^{-1} \mathbf{1}}{\mathbf{1}^T \Sigma^{-1} \mathbf{1}}$$
cannot easily incorporate inequality constraints (such as $w_i \ge 0$), we must shift from pure analytical calculus to Numerical Optimisation to ensure our portfolio is actionable.

In [15]:
from scipy.optimize import minimize

# 1. Define the Objective Function (Minimise Portfolio Variance)
def portfolio_variance(weights, sigma):
    return weights.T @ sigma @ weights

# 2. Define the Constraints
# Equality constraint: Sum of weights must equal 1 (w^T 1 = 1)
cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

# Inequality constraint (Bounds): Each weight must be between 0 and 1
bounds = tuple((0, 1) for _ in range(len(tickers)))

# 3. Initial Guess (Equal weights)
init_guess = [1/len(tickers)] * len(tickers)

# 4. Run the Solver
opt_results = minimize(portfolio_variance, init_guess, args=(sigma,),
                       method='SLSQP', bounds=bounds, constraints=cons)

# 5. Extract the "Realistic" Weights
constrained_weights = pd.Series(opt_results.x, index=tickers)

print("--- Constrained Weights (Long-Only) ---")
print(constrained_weights.round(4))



--- Constrained Weights (Long-Only) ---
SHEL.L    0.0000
BP.L      0.3434
HSBA.L    0.4099
VOD.L     0.2467
dtype: float64


## 8. Analysis of Results

An interesting observation from the numerical optimisation is that the model assigns a weight of 0.0000 to Shell (SHEL.L). While this might seem counter-intuitive at first, it is mathematically and financially logical when considering the structure of our covariance matrix $\Sigma$.

#### The Role of Correlation
The Global Minimum Variance (GMV) portfolio seeks to minimise $\mathbf{w}^T \Sigma \mathbf{w}$. When two assets are highly correlated, they provide redundant risk exposure. In the case of SHEL.L and BP.L:
* Both belong to the Integrated Oil & Gas sector.
* They are influenced by the same underlying risk factors (crude oil prices, global energy demand, and geopolitical stability).
* Their correlation $\rho_{i,j}$ is typically very close to $1$.

#### Mathematical Displacement
If two assets have similar expected variances but high positive correlation, the optimiser will "pick a winner" by allocating to the one that offers the slightly better risk-adjusted profile for the overall portfolio.

In this specific calculation:
1. Redundancy: Including both would not significantly diversify the portfolio's total variance.
2. Optimisation Logic: The optimiser displaced Shell because BP likely exhibited a slightly lower idiosyncratic risk or better covariance properties with the remaining assets (HSBA.L and VOD.L) during this specific period.

This result confirms that the Long-Only constraint is working correctly—rather than taking a short position in Shell to hedge BP, the model simply eliminates the redundant asset to achieve the lowest possible risk floor.

## 9. Out-of-Sample Testing: The "Real World" Check

A common trap in quantitative finance is Overfitting: creating a portfolio that was "perfect" for 2024 but fails in 2025. To validate our model, we must test it on data the optimiser has never seen.

In this section, we compare two strategies over a 1-month "Future" window:
1. The Optimised GMV Portfolio: Using the constrained weights $(\mathbf{w}^*_{long-only})$ derived from our numerical solver.
2. The Naive $1/N$ Portfolio: An equal-weight strategy ($25\%$ split across all assets), which serves as our benchmark.

#### Mathematical Comparison of Risk
We evaluate success by comparing the Annualised Volatility $(\sigma_p)$ of both portfolios:
$$\sigma_p = \text{Std}(R_p) \times \sqrt{252}$$

If our optimisation was successful, the Optimised portfolio should exhibit lower volatility than the Equal-Weight split, proving that accounting for the covariance structure $(\Sigma)$ provides a superior risk-adjusted outcome than naive diversification.

In [16]:
# 1. Download "Future" data (Out-of-sample window)
# We test the strategy on the most recent available data
test_data = yf.download(tickers, start="2025-09-01", end="2025-10-01", auto_adjust=True)['Close']
test_returns = np.log(test_data / test_data.shift(1)).dropna()

# 2. Calculate daily performance for both strategies
# Optimised Weights vs Equal Weights (0.25 each)
portfolio_returns_opt = (test_returns * constrained_weights).sum(axis=1)
portfolio_returns_eq  = (test_returns * 0.25).sum(axis=1)

# 3. Compare the Volatility (Standard Deviation)
# Annualised by multiplying daily std by the square root of 252 trading days
vol_opt = portfolio_returns_opt.std() * np.sqrt(252)
vol_eq  = portfolio_returns_eq.std() * np.sqrt(252)

print(f"--- 1-Month Out-of-Sample Performance ---")
print(f"Annualised Risk (Optimised): {vol_opt:.2%}")
print(f"Annualised Risk (Equal Weight): {vol_eq:.2%}")



[*********************100%***********************]  4 of 4 completed

--- 1-Month Out-of-Sample Performance ---
Annualised Risk (Optimised): 9.22%
Annualised Risk (Equal Weight): 9.85%


## 10. Conclusion and Final Analysis

### 10.1 Summary of Results
In this investigation, we transitioned from a theoretical Lagrangian derivation to a practical Numerical Optimisation. By testing our Global Minimum Variance (GMV) portfolio against a naive $1/N$ strategy, we observed the following:

* Risk Reduction: The optimised portfolio successfully utilised the covariance structure $\Sigma$ to achieve a lower annualised volatility than the equal-weight benchmark.
* Asset Selection: The optimiser intuitively excluded Shell (SHEL.L) due to its high correlation with BP (BP.L), confirming that diversification is not merely about the number of assets, but about the *uniqueness* of their risk profiles.

### 10.2 The Challenge of Estimation Error
While our 1-month test showed a clear advantage, it is important to acknowledge the role of Estimation Error.
Modern Portfolio Theory (MPT) is highly sensitive to the inputs:
1. Stationarity: We assume that correlations from 2023-2024 will persist into 2025. In reality, market regimes shift.
2. Sample Bias: A 1-month testing window is a "snapshot." Over a longer horizon, the "optimal" weights may drift as sector volatilities change.



### 10.3 Final Thoughts
This project demonstrates that while analytical calculus provides the foundation for portfolio theory, numerical methods are required to bridge the gap to real-world investing. By enforcing a Long-Only constraint, we transformed a mathematical abstraction into a physically executable investment strategy that effectively lowered the risk floor for my UK equity holdings.